# VectorTrade Model Training

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import  LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
from utils import FEATURE, TICKER, calc_rsi, predict

In [ ]:
#Loading the Data from yahoo finance
def load_data(ticker, period = '5y'):
    data = yf.download(ticker, period=period, auto_adjust=True)
    data.columns.names = [None, None]
    data.columns = data.columns.get_level_values(0)
    data.reset_index(inplace=True)
    data = data.dropna()
    return data

#creates multiple features
def features(df):
    #we use multiple periods/time-frames to get a holistic picture of the stock.
    #short term momentum
    df["returns_1d"] = df["Close"].pct_change(1)
    df["returns_5d"] = df["Close"].pct_change(5)
    df["returns_10d"] = df["Close"].pct_change(10)

    #moving avg to show actual trend and ignore noises
    df["sma_20"] = df["Close"].rolling(20).mean()
    df["sma_50"] = df["Close"].rolling(50).mean()

    #is the stock undervalued or overvalued
    df["close_to_sma20"] = df["Close"]/df["sma_20"]
    df["close_to_sma50"] = df["Close"]/df["sma_50"]

    #turbulent the share has been
    df["volatility_10d"] = df["returns_1d"].rolling(10).std()

    #volume ratio: determine if spike/drop is in relation to volume
    df["volume_ratio"] = df["Volume"] / df["Volume"].rolling(20).mean()

    #per day candle
    df["hl_range"] = (df["High"] - df["Low"])/ df["Close"]

    #add rsi - very imp feature
    df["rsi_14"] = calc_rsi(df["Close"])
    df = df.dropna()
    return df

#Main label 1-> buy | 0-> sell
def create_labels(df):
    df["target"] = np.where(df["Close"] < df["Close"].shift(-1), 1, 0)
    df.dropna(inplace=True)
    return df

#Modified Data Split using TimeSeriesSplit
def split_data(df):    
    X = df[FEATURE]
    y = df["target"]

    #test_train_split can be use if shuffle=False but wont help you know how much past 
    # is good enough to analyse (gives only 1 window).
    #use time_series_split to maintain the chronological order(row 0 came before row 1)
    time = TimeSeriesSplit(n_splits=5) # generally used but can be anything (5 windows)
    splits = list(time.split(X, y))
    train_idx, test_idx = splits[-1]

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]

    #safety check 
    X_train = X_train.replace([np.inf, -np.inf], np.nan).dropna()
    y_train = y_train[X_train.index]
    X_test = X_test.replace([np.inf, -np.inf], np.nan).dropna()
    y_test = y_test[X_test.index]
    return X_train, X_test, y_train, y_test

#Comparing Models to select right one
def comp_model(df):  
    X_train, X_test, y_train, y_test = split_data(df)

    models = {"RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
              "GradientBoost": GradientBoostingClassifier(n_estimators=100, random_state=42),
              "LogisticRegression": Pipeline([
                  ("scale", StandardScaler()),
                  ("model", LogisticRegression(class_weight='balanced', random_state=42))
              ])
            }
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        print(f"{name} : {accuracy_score(y_test, pred)}")
        
#chose RandomForest
def train_model(df):
    X_train, X_test, y_train, y_test = split_data(df)
    rfc = RandomForestClassifier(n_estimators=100, random_state=42)
    rfc.fit(X_train, y_train)

    predict_rfc = rfc.predict(X_test)
    print(classification_report(y_test, predict_rfc))
    return rfc, predict_rfc

#feature importance
def feature_imp(model, X_train):
    imp = pd.Series(model.feature_importances_, index = X_train.columns).sort_values(ascending=False)
    print(imp)

### Notes
First we had 8 features (excluding rsi), Accuracy Scores:
1. Random Forest: ~0.507
2. Gradient Boost: ~0.488
3. Logistic Regression: ~0.473

Poor Accuracy rates through out all the models so we decided to add RSI.

Accuracy after Adding RSI (9th feature):
1. Random Forest : ~0.488
2. Gradient Boost : ~0.483
3. Logistic Regression : ~0.493 

(just got worse)

We can tell this added more noise to our models than solve the problem. So lets stick with Random Forest and apply feature importance. (selecting features that are useful and excluding rest)

From the feature importance notice that all features are clustered between 0.082 - 0.1 range. We cant really conclude what feature is dominating so best we get is ~50% accuracy. This also explains why no one becomes rich overnight.!!

**Observation:** 
1. More features doesnt guarantee better model.
2. Best Accuracy ~50%, slim chance of making profits.

**Summary of Phase 1:**
load_data()      -> pulls and cleans real market data
features()       -> engineers 9 meaningful signals
create_labels()  -> defines the prediction target
split_data()     -> chronologically safe train/test split (inside utils.py)
comp_model()     -> model selection comparison
train_model()    -> Random Forest classifier (the better one)
feature_imp()    -> diagnoses what the model learned
predict()        -> interface for Phase 2:engine.py (inside utils.py)

In [ ]:
ticker = TICKER
df = load_data(ticker)
df.to_csv(f"../data/price/{ticker}_price.csv", index=False)

In [ ]:
df = features(df)
df = create_labels(df)
X_train, X_test, y_train, y_test = split_data(df)
# comp_model(df)
# feature_imp(model_rfc, X_train) 
model_rfc, predict_rfc = train_model(df)
pred_data = predict(model_rfc, X_test, df)
pred_data.to_csv(f"../data/pred/{ticker}_pred.csv", index=False) 

In [ ]:
joblib.dump(model_rfc, f"../models/{ticker}_model.pkl")